### Sample Code

https://cookbook.openai.com/examples/vector_databases/redis/using_redis_for_embeddings_search

#### Setup

In [3]:
import openai

from typing import List, Iterator
import pandas as pd
import numpy as np
import os
import wget
from ast import literal_eval

# Redis client library for Python
import redis

# I've set this to our new embeddings model, this can be changed to the embedding model of your choice
EMBEDDING_MODEL = "text-embedding-3-small"

# Ignore unclosed SSL socket warnings - optional in case you get these errors
import warnings

warnings.filterwarnings(action="ignore", message="unclosed", category=ResourceWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning) 

#### Load Data

In [4]:
embeddings_url = 'https://cdn.openai.com/API/examples/data/vector_database_wikipedia_articles_embedded.zip'

# The file is ~700 MB so this will take some time
# wget.download(embeddings_url)

In [5]:
import zipfile
with zipfile.ZipFile("vector_database_wikipedia_articles_embedded.zip","r") as zip_ref:
    zip_ref.extractall("../data")

In [6]:
article_df = pd.read_csv('../data/vector_database_wikipedia_articles_embedded.csv')

In [7]:
article_df.head()

,id,url,title,text,title_vector,content_vector,vector_id
0,1,https://simple.wikipedia.org/wiki/April,April,April is the fourth month of the year in the J...,"[0.001009464613161981, -0.020700545981526375, ...","[-0.011253940872848034, -0.013491976074874401,...",0
1,2,https://simple.wikipedia.org/wiki/August,August,August (Aug.) is the eighth month of the year ...,"[0.0009286514250561595, 0.000820168002974242, ...","[0.0003609954728744924, 0.007262262050062418, ...",1
2,6,https://simple.wikipedia.org/wiki/Art,Art,Art is a creative activity that expresses imag...,"[0.003393713850528002, 0.0061537534929811954, ...","[-0.004959689453244209, 0.015772193670272827, ...",2
3,8,https://simple.wikipedia.org/wiki/A,A,A or a is the first letter of the English alph...,"[0.0153952119871974, -0.013759135268628597, 0....","[0.024894846603274345, -0.022186409682035446, ...",3
4,9,https://simple.wikipedia.org/wiki/Air,Air,Air refers to the Earth's atmosphere. Air is a...,"[0.02224554680287838, -0.02044147066771984, -0...","[0.021524671465158463, 0.018522677943110466, -...",4


In [8]:
# Read vectors from strings back into a list
article_df['title_vector'] = article_df.title_vector.apply(literal_eval)
article_df['content_vector'] = article_df.content_vector.apply(literal_eval)

# Set vector_id to be a string
article_df['vector_id'] = article_df['vector_id'].apply(str)

In [9]:
article_df.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id              25000 non-null  int64 
 1   url             25000 non-null  object
 2   title           25000 non-null  object
 3   text            25000 non-null  object
 4   title_vector    25000 non-null  object
 5   content_vector  25000 non-null  object
 6   vector_id       25000 non-null  object
dtypes: int64(1), object(6)
memory usage: 1.3+ MB


### Redis

#### Setup

In [15]:
import redis
from redis.commands.search.indexDefinition import (
    IndexDefinition,
    IndexType
)
from redis.commands.search.query import Query
from redis.commands.search.field import (
    TextField,
    VectorField
)

REDIS_HOST =  "localhost"
REDIS_PORT = 6379
REDIS_PASSWORD = "" # default for passwordless Redis

# Connect to Redis
redis_client = redis.Redis(
    host=REDIS_HOST,
    port=REDIS_PORT,
    password=REDIS_PASSWORD
)
redis_client.ping()

True

#### Creating a Search Index

The below cells will show how to specify and create a search index in Redis. We will

1. Set some constants for defining our index like the distance metric and the index name
2. Define the index schema with RediSearch fields
3. Create the index


In [16]:
# Constants
VECTOR_DIM = len(article_df['title_vector'][0]) # length of the vectors
VECTOR_NUMBER = len(article_df)                 # initial number of vectors
INDEX_NAME = "embeddings-index"                 # name of the search index
PREFIX = "doc"                                  # prefix for the document keys
DISTANCE_METRIC = "COSINE"                      # distance metric for the vectors (ex. COSINE, IP, L2)

In [17]:
# Define RediSearch fields for each of the columns in the dataset
title = TextField(name="title")
url = TextField(name="url")
text = TextField(name="text")
title_embedding = VectorField("title_vector",
    "FLAT", {
        "TYPE": "FLOAT32",
        "DIM": VECTOR_DIM,
        "DISTANCE_METRIC": DISTANCE_METRIC,
        "INITIAL_CAP": VECTOR_NUMBER,
    }
)
text_embedding = VectorField("content_vector",
    "FLAT", {
        "TYPE": "FLOAT32",
        "DIM": VECTOR_DIM,
        "DISTANCE_METRIC": DISTANCE_METRIC,
        "INITIAL_CAP": VECTOR_NUMBER,
    }
)
fields = [title, url, text, title_embedding, text_embedding]

In [18]:
# Check if index exists
try:
    redis_client.ft(INDEX_NAME).info()
    print("Index already exists")
except:
    # Create RediSearch Index
    redis_client.ft(INDEX_NAME).create_index(
        fields = fields,
        definition = IndexDefinition(prefix=[PREFIX], index_type=IndexType.HASH)
    )

#### Load Documents into Index

In [19]:
def index_documents(client: redis.Redis, prefix: str, documents: pd.DataFrame):
    records = documents.to_dict("records")
    for doc in records:
        key = f"{prefix}:{str(doc['id'])}"

        # create byte vectors for title and content
        title_embedding = np.array(doc["title_vector"], dtype=np.float32).tobytes()
        content_embedding = np.array(doc["content_vector"], dtype=np.float32).tobytes()

        # replace list of floats with byte vectors
        doc["title_vector"] = title_embedding
        doc["content_vector"] = content_embedding

        client.hset(key, mapping = doc)

In [20]:
index_documents(redis_client, PREFIX, article_df)
print(f"Loaded {redis_client.info()['db0']['keys']} documents in Redis search index with name: {INDEX_NAME}")

Loaded 25000 documents in Redis search index with name: embeddings-index


#### Running Search Queries

In [30]:
def search_redis(
    openai_client: openai.OpenAI,
    redis_client: redis.Redis,
    user_query: str,
    index_name: str = "embeddings-index",
    vector_field: str = "title_vector",
    return_fields: list = ["title", "url", "text", "vector_score"],
    hybrid_fields = "*",
    k: int = 20,
) -> List[dict]:

    # Creates embedding vector from user query
    embedded_query = openai_client.embeddings.create(input=user_query,
                                            model=EMBEDDING_MODEL,
                                            ).data[0].embedding

    # Prepare the Query
    base_query = f'{hybrid_fields}=>[KNN {k} @{vector_field} $vector AS vector_score]'
    query = (
        Query(base_query)
         .return_fields(*return_fields)
         .sort_by("vector_score")
         .paging(0, k)
         .dialect(2)
    )
    params_dict = {"vector": np.array(embedded_query).astype(dtype=np.float32).tobytes()}

    # perform vector search
    results = redis_client.ft(index_name).search(query, params_dict)
    for i, article in enumerate(results.docs):
        score = 1 - float(article.vector_score)
        print(f"{i}. {article.title} (Score: {round(score ,3) })")
    return results.docs

In [31]:
# For using OpenAI to generate query embedding
from dotenv import load_dotenv
load_dotenv()
open_api_key = os.getenv("OPENAI_API_KEY")
openai_client = openai.OpenAI(api_key=open_api_key)

In [36]:
results = search_redis(openai_client, redis_client, 'modern art in Europe', k=10)

0. General Dynamics F-16 Fighting Falcon (Score: 0.034)
1. Mikoyan-Gurevich MiG-17 (Score: 0.033)
2. The Good, the Bad and the Ugly (Score: 0.028)
3. Genestrerio (Score: 0.028)
4. Mikoyan-Gurevich MiG-15 (Score: 0.026)
5. Musical genre (Score: 0.025)
6. Mikoyan-Gurevich MiG-21 (Score: 0.025)
7. Edvard Grieg (Score: 0.024)
8. Licensed to Ill (Score: 0.023)
9. Grumman F4F Wildcat (Score: 0.023)


In [37]:
results

[Document {'id': 'doc:74703', 'payload': None, 'vector_score': '0.965881347656', 'title': 'General Dynamics F-16 Fighting Falcon', 'url': 'https://simple.wikipedia.org/wiki/General%20Dynamics%20F-16%20Fighting%20Falcon', 'text': 'The General Dynamics F-16 Fighting Falcon is a one-engine multirole combat aircraft. It was originally designed and built by General Dynamics, which is now part of Lockheed Martin. The United States Air Force (USAF) ordered the F-16 in 1972. The airplane first flew four years later in 1976. Over 4,600 F-16s have been built since then. The airplane has been used by more than 25 air forces around the world. It is also used by the Thunderbirds, the USAF\'s aerobatics team. As of 2015, there are more F-16s in military use than any other fixed-wing aircraft.\n\nCapabilities \nThe F-16 is able to fly at twice the speed of sound. It is armed with a M61 Vulcan Gatling gun and has eleven places where weapons or other equipment can be attached. It was the first producti

In [39]:
results = search_redis(openai_client, redis_client, 'Famous battles in Scottish history', vector_field='content_vector', k=10)

0. 585 BC (Score: 0.048)
1. Order of the British Empire (Score: 0.045)
2. 40s BC (Score: 0.042)
3. Order of the Bath (Score: 0.041)
4. Julius Caesar (Score: 0.041)
5. The Convent (Gibraltar) (Score: 0.04)
6. Washington's Birthday (Score: 0.038)
7. 480 (Score: 0.038)
8. 32 (Score: 0.036)
9. 14 BC (Score: 0.036)


#### Hybrid Queries with Redis

In [40]:
def create_hybrid_field(field_name: str, value: str) -> str:
    return f'@{field_name}:"{value}"'

In [42]:
# search the content vector for articles about famous battles in Scottish history and only include results with Scottish in the title
results = search_redis(openai_client,
                       redis_client,
                       "Famous battles in Scottish history",
                       vector_field="title_vector",
                       k=5,
                       hybrid_fields=create_hybrid_field("title", "Scottish")
                       )

0. List of Scottish monarchs (Score: -0.009)
1. Scottish Premier League (Score: -0.01)
2. Scottish Socialist Party (Score: -0.011)
3. First War of Scottish Independence (Score: -0.016)
4. Scottish Football League (Score: -0.017)


In [43]:
# run a hybrid query for articles about Art in the title vector and only include results with the phrase "Leonardo da Vinci" in the text
results = search_redis(openai_client,
                       redis_client,
                       "Art",
                       vector_field="title_vector",
                       k=5,
                       hybrid_fields=create_hybrid_field("text", "Leonardo da Vinci")
                       )

# find specific mention of Leonardo da Vinci in the text that our full-text-search query returned
mention = [sentence for sentence in results[0].text.split("\n") if "Leonardo da Vinci" in sentence][0]
mention

0. Angel (Score: 0.007)
1. The Da Vinci Code (Score: 0.004)
2. May 28 (Score: -0.0)
3. Po (river) (Score: -0.001)
4. Leonardo da Vinci (Score: -0.001)


'The same cherubim creatures were said to be cast in gold on top of the Ark of the Covenant. Casting metal is one of the oldest forms of artwork, and was attempted by Leonardo da Vinci.'